# 02 · Agent Graph와 Tool Calling
이 노트북 전용 Agent Loop가 요청을 관찰·라우팅한 뒤, 선택된 전문 Agent가 필요할 때만 할당된 읽기 전용 Tool을 호출합니다. 실행 뒤 Evaluation Middleware가 Tool audit과 결과 충족도를 검토하고 필요할 때만 최대 3회 보완합니다.

**학습 목표:** 공개 코어 API `run_food_mcp_loop()`로 02 Agent Loop의 Tool Calling 결과와 trace를 확인합니다.  
**실행 경계:** 별도 MCP 서버나 별도 planner loop는 사용하지 않으며, 읽기 전용 Tool만 제공하고 외부 쓰기 작업은 실행하지 않습니다.  

**환경 변수:** 실제 AI Core LLM을 사용하므로 루트 `.env` 설정이 필요합니다.

In [9]:
from pathlib import Path
import sys
from IPython.display import HTML, display

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
sys.path.insert(0, str(project_root / '4_agent_patterns'))

from src.agent_loop_mcp.ui import EVALUATED_WORKFLOW_GRAPH_SVG

display(HTML(EVALUATED_WORKFLOW_GRAPH_SVG))


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI: 02 독립 Agent Loop + Tool Calling</h2>
<p style="margin: 7px 0;">02 전용 대화창·상태 카드·세로 타임라인을 사용합니다. Tool이 실제로 선택되면 해당 Agent 카드에서 종류, 입력, 읽기 전용 결과를 확인하고, Evaluation Middleware의 승인·보완 판정을 확인합니다. 보완은 최대 3회입니다.</p>
  </div>
</div>


In [2]:
import importlib

import src.agent_loop_mcp.ui as loop_ui_module
loop_ui_module = importlib.reload(loop_ui_module)
if 'loop_ui' in globals():
    loop_ui.close()
loop_ui = loop_ui_module.launch_loop_ui()


Closing server running on port: 7868
* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


In [7]:
# Marketing HTML Agent가 생성한 문서 미리보기
from html import escape
from src.agent_loop_mcp.tools import latest_marketing_html_draft

latest_ui_draft = latest_marketing_html_draft()
graph_tool_calls = result.get('tool_calls', []) if 'result' in globals() else []
html_draft = (latest_ui_draft or {}).get('html') or next(
    (call.get('result', {}).get('html') for call in graph_tool_calls
     if call.get('name') == 'generate_marketing_html'),
    None,
)
if html_draft:
    display(HTML(f'<iframe title="Marketing HTML preview" sandbox="" style="width:100%;height:420px;border:1px solid #d9e2f3;border-radius:8px;background:white" srcdoc="{escape(html_draft, quote=True)}"></iframe>'))
else:
    print('같은 노트북 커널에서 UI 또는 Graph 실행으로 생성된 Marketing HTML이 없습니다.')


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>02 Agent Loop의 실행과 Agent별 스트리밍 카드를 확인한다.</li><li>Tool을 선택한 Agent의 완료 행과 카드 상세, Evaluation Middleware의 Tool audit·보완 판정을 확인한다.</li><li>외부 쓰기 Tool이 없고 보완 평가는 최대 3회임을 확인한다.</li></ul>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 같은 작업 보기</h2>
<p style="margin: 7px 0;">노트북은 구현을 복사하지 않고 공개 코어 API만 호출합니다.</p>
  </div>
</div>


In [ ]:
from src.agent_loop_mcp.lab_core import (
    list_food_mcp_tools,
    run_food_mcp_loop_steps,
)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Agent별 Tool 계약 확인</h2><p style="margin: 7px 0;">02 Agent Loop의 각 전문 Agent에 할당된 Function, API, Agent-as-a-Tool 계약을 공개 API로 확인합니다.</p></div></div>


In [ ]:
tool_catalog = list_food_mcp_tools()
for tool in tool_catalog:
    print(f"{tool['agent_label']} | {tool['name']} ({tool['type']}) | read_only={not tool['write']}")
    print(f"  └ {tool['description']}")


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. Cell 단위 Agent Loop 실행</h2><p style="margin: 7px 0;">Observer, Router, Planning Agent, 실행 계획, 다음 Agent를 각각 별도 셀에서 진행합니다. <code>다음 Agent 또는 평가 1회 진행</code> 셀은 실행할 때마다 정확히 한 Agent의 Tool audit 또는 평가 단위 하나만 진행합니다.</p></div></div>


In [ ]:
request = '고단백 닭가슴살 도시락의 원가·영양·식품안전·공급망을 검토하고 출시 권고안을 작성해줘.'
loop_steps = run_food_mcp_loop_steps(request)
result = None
print('Agent Loop session을 준비했습니다. 다음 셀부터 한 단위씩 진행하세요.')


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. Observer와 Router</h2><p style="margin: 7px 0;">두 셀에서 대화 관찰 결과와 routing score를 각각 확인합니다.</p></div></div>


In [ ]:
observer_step = next(loop_steps)
assert observer_step['step'] == 'observer', observer_step['step']
print(observer_step['observation'])


In [ ]:
route_step = next(loop_steps)
assert route_step['step'] == 'route', route_step['step']
print(route_step['decision'])


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. Planning Agent와 실행 계획</h2><p style="margin: 7px 0;">Planning Agent의 공통 브리프와 LLM이 선택한 단계별 Agent를 분리해 확인합니다.</p></div></div>


In [ ]:
planning_step = next(loop_steps)
assert planning_step['step'] == 'agent', planning_step['step']
print(f"{planning_step['agent_label']} ({planning_step['agent_id']})")
print(planning_step['answer'])


In [ ]:
plan_step = next(loop_steps)
assert plan_step['step'] == 'plan', plan_step['step']
for number, stage in enumerate(plan_step['plan']['stages'], start=1):
    print(f"{number}. {', '.join(stage['agent_ids'])}: {stage['objective']}")
print('최종 Agent:', plan_step['plan']['final_agent_id'])


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. 다음 Agent 또는 평가 1회 진행</h2><p style="margin: 7px 0;">이 셀을 다시 실행할 때마다 다음 Agent 하나의 응답·Tool audit, 또는 Evaluation Middleware의 한 단위만 진행됩니다. <code>complete</code>가 나올 때까지 반복하세요.</p></div></div>


In [ ]:
next_step = next(loop_steps)
if next_step['step'] == 'agent':
    phase = '보완' if next_step['rework'] else '실행'
    print(f"[{phase}] {next_step['agent_label']} ({next_step['agent_id']})")
    print(next_step['answer'])
    for call in next_step['tool_calls']:
        tool_result = call.get('result', {})
        boundary = tool_result.get('execution_boundary') if isinstance(tool_result, dict) else None
        print(f"  [Tool] {call['name']} | arguments={call.get('arguments', {})} | boundary={boundary or '읽기 전용'}")
elif next_step['step'] == 'evaluation':
    review = next_step['evaluation']
    print(f"[Evaluation {review['loop']}] {review['verdict']}: {review['reason']}")
elif next_step['step'] == 'complete':
    result = next_step['result']
    print(f"[complete] tools={len(result['tool_calls'])} | review_required={result['review_required']}")
    print(result['result'])
else:
    print(next_step)
